# Session 08 — Window Functions
**Topics:** Window Spec · Ranking · lag/lead · Running Totals · Latest Record per Customer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Session08").master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

---
## 1 · Defining a Window Spec

A **window spec** defines the *scope* over which a function is computed — without collapsing rows like `groupBy` does.

```python
# Core syntax
window = Window.partitionBy("col1").orderBy("col2")

# partitionBy → like GROUP BY  — resets the window per group
# orderBy     → row order within each partition
```

In [0]:
# Dataset: monthly sales per employee per department
data = [
    ("Alice",   "Engineering", "2024-01", 82000),
    ("Alice",   "Engineering", "2024-02", 91000),
    ("Alice",   "Engineering", "2024-03", 87000),
    ("Bob",     "Engineering", "2024-01", 74000),
    ("Bob",     "Engineering", "2024-02", 69000),
    ("Bob",     "Engineering", "2024-03", 78000),
    ("Charlie", "Marketing",   "2024-01", 55000),
    ("Charlie", "Marketing",   "2024-02", 61000),
    ("Charlie", "Marketing",   "2024-03", 58000),
    ("Diana",   "Marketing",   "2024-01", 67000),
    ("Diana",   "Marketing",   "2024-02", 71000),
    ("Diana",   "Marketing",   "2024-03", 64000),
]
sales_df = spark.createDataFrame(data, ["emp", "dept", "month", "sales"])
sales_df.show(12)

In [0]:
# Window partitioned by dept, ordered by sales descending
# This window is reused across all ranking cells below
dept_window = Window.partitionBy("dept").orderBy(F.desc("sales"))

---
## 2 · row_number(), rank(), dense_rank()

| Function | Ties handled how? | Gaps after ties? |
|---|---|---|
| `row_number()` | Arbitrary — no two rows share the same number | No |
| `rank()` | Tied rows get the same rank | Yes — next rank skips |
| `dense_rank()` | Tied rows get the same rank | No — ranks are consecutive |

In [0]:
# Introduce a tie — Bob and Alice both have sales=82000 in Jan
rank_data = [
    ("Alice",   "Engineering", 91000),
    ("Bob",     "Engineering", 82000),
    ("Charlie", "Engineering", 82000),  # tie with Bob
    ("Diana",   "Engineering", 74000),
    ("Eve",     "Marketing",   71000),
    ("Frank",   "Marketing",   71000),  # tie with Eve
    ("Grace",   "Marketing",   65000),
]
rank_df = spark.createDataFrame(rank_data, ["emp", "dept", "sales"])

w = Window.partitionBy("dept").orderBy(F.desc("sales"))

rank_df.withColumn("row_number",  F.row_number().over(w)) \
       .withColumn("rank",        F.rank().over(w)) \
       .withColumn("dense_rank",  F.dense_rank().over(w)) \
       .orderBy("dept", F.desc("sales")) \
       .show()

**Observe:** Bob & Charlie are tied at 82000 in Engineering.
- `row_number` → 2 and 3 (no ties, arbitrary order)
- `rank`       → both get 2, next rank jumps to 4 (gap)
- `dense_rank` → both get 2, next rank is 3 (no gap)

---
## 3 · lag() and lead()

```python
F.lag("col",  n, default).over(window)   # value n rows BEHIND current row
F.lead("col", n, default).over(window)   # value n rows AHEAD  of current row
```
Used to compare a row with its previous or next row — e.g. month-over-month change.

In [0]:
# Window: per employee, ordered by month
emp_month_window = Window.partitionBy("emp").orderBy("month")

sales_df.withColumn("prev_month_sales", F.lag("sales",  1, 0).over(emp_month_window)) \
        .withColumn("next_month_sales", F.lead("sales", 1, 0).over(emp_month_window)) \
        .withColumn("mom_change",
                    F.col("sales") - F.lag("sales", 1).over(emp_month_window)) \
        .orderBy("emp", "month") \
        .show(12)

**Observe:**
- First row per employee → `prev_month_sales` = 0 (default), `mom_change` = null
- Last row per employee  → `next_month_sales` = 0 (default)
- `lag` with no default returns null for boundary rows

---
## 4 · Running Totals with sum().over(window)

A **running total** accumulates values row by row within a partition.  
Key addition to the window spec: `rowsBetween(Window.unboundedPreceding, Window.currentRow)`

In [0]:
# Running total of sales per employee across months
running_window = Window.partitionBy("emp") \
                       .orderBy("month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_df.withColumn("running_total", F.sum("sales").over(running_window)) \
        .orderBy("emp", "month") \
        .show(12)

In [0]:
# Contrast: running total vs full partition total
full_window    = Window.partitionBy("emp")
running_window = Window.partitionBy("emp").orderBy("month") \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_df.withColumn("total_all_months", F.sum("sales").over(full_window)) \
        .withColumn("running_total",    F.sum("sales").over(running_window)) \
        .withColumn("pct_of_total",
                    F.round(F.col("running_total") / F.col("total_all_months") * 100, 1)) \
        .orderBy("emp", "month") \
        .show(12)

---
## 5 · Practical Use Case — Latest Record per Customer

Real-world scenario: a customer table has multiple rows per customer (updates over time).  
Goal → keep only the **most recent** row per customer.

In [0]:
from pyspark.sql.functions import to_timestamp

customers = [
    (1, "Alice", "alice@old.com",     "2023-01-10 09:00:00"),
    (1, "Alice", "alice@new.com",     "2024-03-15 14:30:00"),  # latest for cust 1
    (1, "Alice", "alice@mid.com",     "2023-09-20 11:00:00"),
    (2, "Bob",   "bob@company.com",   "2023-06-01 08:00:00"),
    (2, "Bob",   "bob@personal.com",  "2024-01-22 16:45:00"),  # latest for cust 2
    (3, "Carol", "carol@domain.com",  "2024-05-30 10:00:00"),  # only record for cust 3
]
cust_df = spark.createDataFrame(customers, ["cust_id", "name", "email", "updated_at"])
cust_df = cust_df.withColumn("updated_at", to_timestamp("updated_at"))
cust_df.orderBy("cust_id", "updated_at").show(truncate=False)

In [0]:
# Step 1: assign row_number per customer, latest updated_at = rank 1
cust_window = Window.partitionBy("cust_id").orderBy(F.desc("updated_at"))

ranked = cust_df.withColumn("rn", F.row_number().over(cust_window))
ranked.orderBy("cust_id", "rn").show(truncate=False)

In [0]:
# Step 2: keep only rn = 1 → one row per customer, the latest
latest_df = ranked.filter(F.col("rn") == 1).drop("rn")
latest_df.show(truncate=False)

---
## Quick Reference

| Function | Window needs `orderBy`? | Typical use |
|---|---|---|
| `row_number()` | ✅ | Unique sequential rank, dedup |
| `rank()` | ✅ | Rank with gaps on ties |
| `dense_rank()` | ✅ | Rank without gaps on ties |
| `lag(col, n)` | ✅ | Previous row value |
| `lead(col, n)` | ✅ | Next row value |
| `sum().over(w)` | Optional | Running total / partition total |
